# agentic_alpha_factor_library_reversal_search_v1

Diagnostic R-G-E-V-D card for local factor-library reversal, sell-composite, and volatility-managed ideas. It retrieves the existing qlib SOXL/SOXX provider through `D.features`, rebuilds selected factor-library signals in `src`, evaluates production-window candidates against `qlib_paper_sim_v1`, records the run with `R.start`, and keeps only slim sidecars in the card root.

In [1]:

from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import pandas as pd
from qlib.workflow import R

ROOT_CANDIDATES = [Path.cwd(), Path('/Users/rziiiii/dev/workspace')]
ROOT = next(
    path
    for base in ROOT_CANDIDATES
    for path in [base, *base.parents]
    if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.qlib_native import QlibProviderConfig, load_qlib_feature_frame
from src.factor import (
    build_factor_library_reversal_frame,
    compact_factor_library_search_summary,
    evaluate_factor_library_reversal_candidates,
)
from src.workflow import load_agentic_alpha_backtest_metric

EXPERIMENT = 'agentic_alpha_factor_library_reversal_search_v1'
CARD = ROOT / 'research' / EXPERIMENT
PROVIDER_URI = (ROOT / 'data' / 'qlib' / 'soxl').resolve()
RECORDER_URI = (CARD / 'qlib_recorder').resolve()
PRODUCTION_CARD = ROOT / 'production' / 'qlib_paper_sim_v1'
FEATURE_START = '2010-03-11'
FEATURE_END = '2026-06-10'
TEST_START = '2019-01-02'
TEST_END = '2026-06-09'
MAX_DRAWDOWN_FLOOR = -20.0

CARD.mkdir(parents=True, exist_ok=True)
if RECORDER_URI.exists():
    shutil.rmtree(RECORDER_URI)
RECORDER_URI.mkdir(parents=True, exist_ok=True)

baseline = load_agentic_alpha_backtest_metric(PRODUCTION_CARD, window='test')

workflow_yaml = f'''experiment: {EXPERIMENT}
card_type: diagnostic_search
qlib:
  provider_uri: {PROVIDER_URI.as_posix()}
  recorder_uri: {RECORDER_URI.as_posix()}
  market: all
  region: us
segments:
  feature_warmup:
    start: {FEATURE_START}
    end: {FEATURE_END}
  production_test:
    start: {TEST_START}
    end: {TEST_END}
rgevd:
  source: https://github.com/JacobDu/agentic-alpha
  stages:
    - research
    - retrieve
    - generate
    - evaluate
    - validate
    - distill
goal_gate:
  baseline_card: production/qlib_paper_sim_v1
  baseline_return_pct: {baseline.return_pct}
  max_drawdown_floor: {MAX_DRAWDOWN_FLOOR}
  require_return_above_baseline: true
artifact_policy:
  policy: slim_card_recorder_first
  machine_entrypoint: summary.json
'''
(CARD / 'workflow.yaml').write_text(workflow_yaml, encoding='utf-8')

external_sources = pd.DataFrame(
    [
        {
            'source_type': 'github',
            'title': 'agentic-alpha R-G-E-V-D workflow',
            'url': 'https://github.com/JacobDu/agentic-alpha',
            'extracted_signal': 'Use retrieve-generate-evaluate-validate-distill loops to close local factor-library ideas with comparable evidence.',
            'retrieval_status': 'workflow_reference',
        },
        {
            'source_type': 'paper',
            'title': 'Volatility-Managed Portfolios',
            'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2659431',
            'extracted_signal': 'Reduce exposure when realized volatility is high; test whether volatility-managed SOXL factor scores preserve return while cutting drawdown.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'paper',
            'title': 'Expected Stock Returns and Variance Risk Premia',
            'url': 'https://academic.oup.com/rfs/article-abstract/22/11/4463/1565787',
            'extracted_signal': 'Volatility and variance-premium state can contain return predictability, motivating RB_VOL_REGIME-style gates.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'arxiv',
            'title': 'Compounding Effects in Leveraged ETFs',
            'url': 'https://arxiv.org/abs/2504.20116',
            'extracted_signal': 'Daily leveraged ETF compounding depends on trend persistence versus mean reversion and volatility clustering.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'twitter',
            'title': 'SOXL drawdown and volatility discussion seed',
            'url': 'https://x.com/ces921/highlights',
            'extracted_signal': 'Treat SOXL as regime-sensitive; reversal entries and sell gates must be judged by drawdown, not signal IC alone.',
            'retrieval_status': 'search_seed',
        },
    ]
)
external_sources.to_csv(CARD / 'external_sources.csv', index=False)

provider_config = QlibProviderConfig(provider_uri=PROVIDER_URI, region='us', market='all', freq='day')
qlib_fields = ['$open', '$high', '$low', '$close', '$volume', '$vix']
qlib_frame = load_qlib_feature_frame(
    provider_config,
    instruments=['SOXL', 'SOXX'],
    fields=qlib_fields,
    start=FEATURE_START,
    end=FEATURE_END,
)
feature_frame = build_factor_library_reversal_frame(qlib_frame, start=TEST_START, end=TEST_END)

candidate_metrics = evaluate_factor_library_reversal_candidates(
    feature_frame,
    baseline_return_pct=baseline.return_pct,
    max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
)
candidate_metrics.to_csv(CARD / 'agentic_alpha_factor_library_reversal_candidate_metrics.csv', index=False)

branches = sorted(candidate_metrics['branch'].unique())
compact_summary = compact_factor_library_search_summary(
    candidate_metrics,
    branches=branches,
    baseline_return_pct=baseline.return_pct,
    max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
)
passes_both = candidate_metrics.loc[candidate_metrics['beats_baseline_return'] & candidate_metrics['passes_drawdown_gate']]

search_summary = {
    'workflow': 'agentic_alpha_factor_library_reversal_search',
    'stage_order': ['research', 'retrieve', 'generate', 'evaluate', 'validate', 'distill'],
    'baseline': {
        'name': baseline.name,
        'return_pct': float(baseline.return_pct),
        'max_drawdown_pct': float(baseline.max_drawdown_pct),
        'benchmark_return_pct': float(baseline.benchmark_return_pct),
        'recorder_id': baseline.recorder_id,
    },
    'goal_achieved': bool(compact_summary['goal_achieved']),
    'best_candidate': compact_summary['best_candidate'],
    'passes_both_count': int(compact_summary['passes_both_count']),
    'branch_results': compact_summary['branch_results'],
    'external_source_counts': external_sources['source_type'].value_counts().sort_index().to_dict(),
    'factor_library': {
        'source_db': 'data/soxl_factor_library.db',
        'selected_features': [
            'RB_RSI6_OVERSOLD',
            'RB_SELL_COMPOSITE',
            'BIAS_10',
            'BIAS_10_MOM',
            'REL_VOLUME_RATIO',
            'PRICE_RATIO_MOM',
            'SOXX_BIAS_10',
            'RB_VOL_REGIME',
        ],
        'candidate_coverage_start': feature_frame['datetime'].min().date().isoformat(),
        'candidate_coverage_end': feature_frame['datetime'].max().date().isoformat(),
        'source_coverage_start': feature_frame.attrs['source_coverage_start'],
        'source_coverage_end': feature_frame.attrs['source_coverage_end'],
        'feature_rows': int(len(feature_frame)),
        'qlib_provider': PROVIDER_URI.as_posix(),
        'qlib_fields': qlib_fields,
        'qlib_feature_rows': int(len(qlib_frame.reset_index())),
    },
}
(CARD / 'agentic_alpha_factor_library_reversal_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')

failure_modes = (
    '# Failure Modes\n\n'
    '- factor_library_reversal_reentry: 反转再入场可以产生很高的局部收益，但高杠杆版本承受深度 SOXL 路径回撤；回撤 gate 内收益远低于生产基准。\n'
    '- factor_library_trend_sell_gate: 趋势持有叠加 sell_score 能把 MaxDD 降到 20% 附近，但需要把平均暴露压低到约十几个百分点，复利不足。\n'
    '- factor_library_vol_managed_score: volatility-managed score 降低了路径风险，但在本窗口内主要通过降暴露达成，return_gate 失败。\n'
    '- data_boundary: 因子库 IC/信号回测是检索证据，不是 promotion 证据；promotion 只看本卡用 qlib provider 和 production baseline 重新计算的 return/DD。\n'
    '- next_action: 若继续寻找，需要更换信息源或模型家族，例如更长历史期权风险、资金流、订单流或非阈值化模型，而不是重复相同 reversal 阈值。\n'
)
(CARD / 'failure_modes.md').write_text(failure_modes, encoding='utf-8')

R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder_id = recorder.id
    R.log_params(
        baseline=baseline.name,
        provider_uri=PROVIDER_URI.as_posix(),
        max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
        feature_start=FEATURE_START,
        feature_end=FEATURE_END,
        test_start=TEST_START,
        test_end=TEST_END,
        candidate_rows=len(candidate_metrics),
    )
    search_summary['recorder_id'] = recorder_id
    (CARD / 'agentic_alpha_factor_library_reversal_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')
    for path in [
        CARD / 'agentic_alpha_factor_library_reversal_candidate_metrics.csv',
        CARD / 'agentic_alpha_factor_library_reversal_search_summary.json',
        CARD / 'external_sources.csv',
        CARD / 'failure_modes.md',
    ]:
        recorder.log_artifact(path.as_posix(), artifact_path='agentic_alpha_factor_library_reversal')
    recorder_local_dir = Path(recorder.get_local_dir())

for transient_name in ['code_cached.txt', 'code_diff.txt', 'code_status.txt']:
    transient_path = recorder_local_dir / 'artifacts' / transient_name
    if transient_path.exists():
        transient_path.unlink()
recorder_artifacts = sorted(path.relative_to(recorder_local_dir).as_posix() for path in recorder_local_dir.rglob('*') if path.is_file())
search_summary['recorder_artifacts'] = recorder_artifacts
(CARD / 'agentic_alpha_factor_library_reversal_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')

summary = {
    'experiment': EXPERIMENT,
    'card_type': 'diagnostic_search',
    'promotion_decision': 'not_promoted' if not search_summary['goal_achieved'] else 'promote_candidate',
    'as_of': TEST_END,
    'data_range': [TEST_START, TEST_END],
    'feature_warmup_range': [FEATURE_START, FEATURE_END],
    'provider_uri': PROVIDER_URI.as_posix(),
    'recorder_uri': RECORDER_URI.as_posix(),
    'recorder_id': recorder_id,
    'recorder_local_dir': recorder_local_dir.as_posix(),
    'recorder_artifacts': recorder_artifacts,
    'market': 'all',
    'region': 'us',
    'baseline': search_summary['baseline'],
    'goal_gate': {'max_drawdown_floor': MAX_DRAWDOWN_FLOOR, 'require_return_above_baseline': True},
    'branch_results': search_summary['branch_results'],
    'search_report': search_summary,
    'external_source_counts': search_summary['external_source_counts'],
    'factor_library': search_summary['factor_library'],
    'qlib_docs': {
        'data': 'https://qlib.readthedocs.io/en/latest/component/data.html',
        'model': 'https://qlib.readthedocs.io/en/latest/component/model.html',
        'workflow': 'https://qlib.readthedocs.io/en/latest/component/workflow.html',
    },
    'qlib_native_api': ['qlib.init', 'qlib.data.D.features', 'qlib.workflow.R.start'],
    'workflow_config': 'workflow.yaml',
    'sidecars': {
        'candidate_metrics': 'agentic_alpha_factor_library_reversal_candidate_metrics.csv',
        'external_sources': 'external_sources.csv',
        'failure_modes': 'failure_modes.md',
        'search_summary': 'agentic_alpha_factor_library_reversal_search_summary.json',
    },
    'artifact_policy': {
        'policy': 'slim_card_recorder_first',
        'workflow_config': 'workflow.yaml',
        'machine_entrypoint': 'summary.json',
        'primary_artifact': f'{EXPERIMENT}.ipynb',
        'kept_sidecars': [
            'agentic_alpha_factor_library_reversal_candidate_metrics.csv',
            'agentic_alpha_factor_library_reversal_search_summary.json',
            'external_sources.csv',
            'failure_modes.md',
        ],
        'recorder_only_artifacts': {
            'predictions': 'artifacts/pred.pkl',
            'labels': 'artifacts/label.pkl',
            'signals': 'artifacts/signals.pkl',
            'portfolio_report': 'artifacts/portfolio_analysis/report_normal_1day.pkl',
            'positions': 'artifacts/portfolio_analysis/positions_normal_1day.pkl',
            'risk_analysis': 'artifacts/portfolio_analysis/port_analysis_1day.pkl',
            'all_model_predictions': 'artifacts/model_selection/all_model_predictions.csv',
        },
        'reproducible_from_recorder': True,
    },
}
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
(CARD / 'conclusion.md').write_text(
    '# 结论\n\n'
    f'- what_ran: 执行 {EXPERIMENT}，从 Github/论文/Arxiv/Twitter 检索到的 R-G-E-V-D 思路映射到本地 factor library 的 reversal、sell_score 和 volatility-managed 分支，并通过 qlib `D.features` 读取 SOXL/SOXX 日线重新评估。\n'
    f'- key_metrics: production baseline return={baseline.return_pct:.2f}%, maxDD={baseline.max_drawdown_pct:.2f}%; 本卡 candidate_rows={len(candidate_metrics)}, passes_both_count={len(passes_both)}。\n'
    f'- regime_window_difference: 特征暖身使用 {FEATURE_START} 到 {FEATURE_END}，production-window 候选覆盖 {feature_frame["datetime"].min().date().isoformat()} 到 {feature_frame["datetime"].max().date().isoformat()}。\n'
    '- promotion_decision: not_promoted，未找到同时满足收益超过现有生产策略且 MaxDD 小于 20% 的候选。\n'
    '- risks: 因子库强 IC 信号在组合层面仍然被 SOXL 路径风险主导；回撤 gate 内的参数主要靠降暴露，而不是发现了新的高收益低回撤结构。\n'
    '- next_action: 下一轮不要继续重复 RSI/BIAS/reversal 阈值，优先转向更长历史的期权风险、资金流、订单流或非阈值化模型族。\n',
    encoding='utf-8',
)
print(json.dumps({'recorder_id': recorder_id, 'passes_both_count': int(len(passes_both)), 'branch_results': search_summary['branch_results']}, indent=2, sort_keys=True))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[80890:MainThread](2026-06-23 17:54:09,820) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[80890:MainThread](2026-06-23 17:54:09,823) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[80890:MainThread](2026-06-23 17:54:09,824) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[80890:MainThread](2026-06-23 17:54:10,846) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name agentic_alpha_factor_library_reversal_search_v1.


[80890:MainThread](2026-06-23 17:54:10,848) INFO - qlib.workflow - [exp.py:258] - Experiment 885873317903939296 starts running ...


[80890:MainThread](2026-06-23 17:54:10,922) INFO - qlib.workflow - [recorder.py:345] - Recorder 2f196ca10d4341a4833f34f2a2c1a1f6 starts running under Experiment 885873317903939296 ...


[80890:MainThread](2026-06-23 17:54:11,004) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{
  "branch_results": {
    "factor_library_reversal_reentry": {
      "best_drawdown_gate_available": true,
      "best_drawdown_gate_candidate": "panic_buy_score_thr0.75_hold1_scale4.0_volcap0.8_none",
      "best_drawdown_gate_max_drawdown_pct": -15.481303436583394,
      "best_drawdown_gate_return_pct": 99.48770741248762,
      "best_return_candidate": "panic_buy_score_thr0.65_hold1_scale10.0_volcapNone_none",
      "best_return_max_drawdown_pct": -9260.209989634768,
      "best_return_pct": 17813869.903124534,
      "coverage_end": "2026-06-09",
      "coverage_start": "2019-01-02",
      "goal_baseline_return_pct": 6570.0232821043,
      "grid_rows": 540,
      "max_drawdown_floor": -20.0,
      "passes_both_count": 0
    },
    "factor_library_trend_sell_gate": {
      "best_drawdown_gate_available": true,
      "best_drawdown_gate_candidate": "trend_score_075_sell0.5_buy0.65_base0.25_boost1.0_volcap0.25",
      "best_drawdown_gate_max_drawdown_pct": -19.06680592698985,
      "b

## Final Chinese Conclusion

The executed cell writes `conclusion.md` with the required Chinese summary: what ran, key metrics, window differences, promotion decision, risks, and next action.